## 留意事項
- 本資料は教育・情報提供を目的としたもので、特定銘柄の売買推奨ではありません。
- バックテスト結果は将来成果を保証しません。
- 実運用は自己責任で行ってください。

# 5/21 先物乖離_Zスコアクロスセクション

> **対象**: 裁量取引していた方がAIを活用して戦略実装・自動化を実践している方<br>
> **目的**: 前回4/23 botter会の宿題「Zスコアを用いたクロスセクション戦略」を検証する<br>
> **データ**: Binance 371銘柄、2021〜2026年、日次OHLCV

![botterkai](botterkai.jpeg)

# botter会とは？
* botterを継続するためのきっかけ作り
* 進捗共有 → 先生からFB→ 宿題 → 検証
* 「真のbotter」をめざしてがんばろう！



# 前回までのおさらい

### 条件
   * クロスセクション戦略
   * 先物乖離指標ベース

| 項目 | ロジック |
|---|---|
| 戦略タイプ | **クロスセクション（ショートのみ）** |
| 目的 | 先物プレミアム（コンタンゴ）が大きい銘柄の翌日収束を狙う |
| シグナル | `basis = (先物終値 - 現物終値) / 現物終値` |
| エントリー | 各日、`basis > 0.1%` かつ `30日平均先物出来高 < 50M` の銘柄を抽出し、`basis` 大きい順に **Top20** をショート |
| BTCフィルター | `BTCの30日リターン > +10%` の日は **取引しない** |
| イグジット | **翌日クローズ**（1日保有）、毎日リバランス |
| ポジション配分 | Top20を等ウェイト（各銘柄同額） |
| 手数料 | 片道 `0.04%`（往復0.08%想定）。日次損益は概念的に `-平均(next_ret) - fee` |
| 採用パラメータ | `THR=0.001`, `VOL_THR=50e6`, `BTC_THR=0.10`, `TOP_N=20` |
| 評価期間 | IS: 2021-01-01〜2022-12-31 / OOS: 2023-01-01〜2026-12-31 |


![20260521_botterkai](20260521_botterkai.png)

**ほへと先生からの指摘**

> 「銘柄によって『常にショートヘッジが入っていて乖離がマイナスになりやすい』などの特性がある。  
> 生データ同士を比較しても意味がない。  
> 必ず直近のデータで **標準化（Zスコア化）** を行うこと。」


# 今回の宿題
### 条件
* Zスコア化
   * 銘柄ごとにbasisを14日ローリングでZスコア化し、クロスセクション戦略を正式に検証する
* IS-OOS検証
   * IS: 2021-01-01〜2022-12-31 / OOS: 2023-01-01〜2026-12-31


![20260521_botterkai2](20260521_botterkai2.png)

## 検証結果サマリー
### 何をしたか？

| Step | 戦略 | 改善内容 | OOS Sharpe |
|------|------|---------|-----------|
| 基本 | 生basis版（前回のまま） | ベースライン | -0.275 |
| 1 | **Zスコア化**（今回の宿題） | 銘柄間の公平な比較 | +0.108 |
| 2 | +BTCスイッチング | 相場の方向性に合わせてロング/ショートを切り替え | +0.341 |
| 3 | +コンタンゴフィルター（最終版） | 市場全体が弱気の時はロングしない | **+0.90** |

### 詳細

| Step | 何をしたか | 判定ルール | ねらい |
|------|------|------|------|
|  生basis版 | `basis` をそのまま銘柄横断比較し、上位10ショート・下位10ロングを日次で実行 | 毎日リバランス、翌日決済 | ベースライン確認 |
|  Zスコア化 | 銘柄ごとに `basis` を14日ローリングで標準化してランキング | `zscore = (basis - 14日平均) / 14日標準偏差` | 銘柄間の比較を公平化 |
| + BTCスイッチングフィルター | 相場方向でロング/ショートを切替 | `BTC30日リターン > +7%` → ロングのみ、`< -7%` → ショートのみ、`±7%以内` → スキップ | BTC相場の地合いに逆らわず、片側だけを拾う |
| + コンタンゴフィルター | ロング実行日を市場全体の強弱で追加フィルタ | コンタンゴ比率（ basis>0 の銘柄数 / 全銘柄数）が`20%未満`の日はロングしない | 市場全体が弱気相場でのロングを回避（OOSの負けがロング側で起きやすかったから） |

> 共通条件: 手数料0.04%/片道、対象は上位/下位10銘柄、日次リバランス。


---
# 📕 参考

## クロスセクション先物乖離戦略とは

### 先物乖離（basis）とは

暗号資産の先物価格と現物価格の差を「乖離率（basis）」と呼ぶ。

```
basis = (先物価格 - 現物価格) / 現物価格
```

- **プラス（コンタンゴ）**：先物が現物より高い → 市場が強気、先物買いが多い
- **マイナス（バックワーデーション）**：先物が現物より安い → 市場が弱気、先物売りが多い

この乖離は、時間が経つと収束する性質がある（先物の限月交代・裁定取引による）。

---

### クロスセクション戦略とは

371銘柄を毎日横断的に比較し、**乖離が大きい銘柄はショート、小さい銘柄はロング**する戦略。

```
毎日:
  全銘柄の basis をZスコア化（銘柄ごとの相対的な割高・割安を算出）
        ↓
  Zスコア 上位10銘柄 → ショート（先物が割高 → 収束を狙う）
  Zスコア 下位10銘柄 → ロング  （先物が割安 → 収束を狙う）
        ↓
  翌日に決済・毎日銘柄を組み替える
```

**なぜZスコア化が必要か？**  
銘柄ごとに乖離の「普通の水準」が違う。生basisで比較すると「常に乖離が高い銘柄」だけが毎日ランク上位になってしまう。  
Zスコア化することで「その銘柄にとって今日が特に割高か」という公平な比較ができる。



---
---
## 目次

1. [環境セットアップ・データ読み込み](#section0)
2. [仮説](#section-hypothesis)
3. [Step1: 生basis版（ベースライン）](#section1)
4. [Step2: Zスコア化の効果](#section2)
5. [Step3: BTCスイッチング + IS/OOS検証](#section3)
6. [Step4: コンタンゴフィルター](#section4)
7. [まとめ](#summary)

---
### 🔧 環境セットアップ・データ読み込み

**何をするか**
- 必要なライブラリをインポートする
- 分析全体で使うパラメータを定義する（Zスコア窓・手数料・BTC判定閾値など）
- Binanceの日次データ（371銘柄、2021〜2026年）を読み込む
- `basis`（先物と現物の乖離率）と`next_ret`（翌日リターン）を計算する

In [ ]:
# <a id="section0"></a>
# ============================================================
# 0. 環境セットアップ・データ読み込み
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "#f8f9fa"
plt.rcParams["font.family"] = ["Hiragino Sans", "sans-serif"]

# ---- パラメータ ----
Z_WINDOW = 14  # Zスコア計算の窓
TOP_N = 10  # ロング/ショートそれぞれ何銘柄
FEE = 0.0004  # 手数料 0.04%/片道
BTC_WIN = 30  # BTC方向判定の窓
BUFFER = 0.07  # バッファ ±7%
CONTANGO_THR = 0.20  # コンタンゴフィルター閾値 20%
IS_END = "2022-12-31"
OOS_START = "2023-01-01"


# ---- ヘルパー関数 ----
def sharpe(r):
    return r.mean() / r.std() * np.sqrt(365) if r.std() > 0 else 0


def maxdd(r):
    c = r.cumsum()
    return (c - c.cummax()).min() * 100


def winrate(r):
    return (r > 0).mean() * 100


# ---- データ読み込み ----
candidates = [Path("master_2021.csv"), Path("./data/master_2021.csv")]
csv_path = next((p for p in candidates if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError(
        "master_2021.csv が見つかりません。ノートブックと同じフォルダ、または data/ に配置してください。"
    )
df = pd.read_csv(csv_path, parse_dates=["timestamp"])
df = df.rename(
    columns={
        "timestamp": "date",
        "spot_close": "sp_close",
        "spot_vol": "sp_vol",
        "future_close": "fu_close",
        "future_vol": "fu_vol",
    }
)
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_localize(None)
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)

# ---- 基本指標の計算 ----
df["basis"] = (df["fu_close"] - df["sp_close"]) / df["sp_close"]
df["next_ret"] = df.groupby("symbol")["fu_close"].transform(lambda x: x.pct_change().shift(-1))

print(f"データ件数: {len(df):,}行")
print(f"銘柄数: {df['symbol'].nunique()}銘柄")
print(f"期間: {df['date'].min().date()} 〜 {df['date'].max().date()}")

---
### Zスコアの計算

**何をするか**
- 銘柄ごとに `basis` を14日ローリングでZスコア化する
  ```
  zscore = (basis - μ_14d) / σ_14d
  ```
- BTC30日リターン（相場の方向性シグナル）を計算する
- コンタンゴ比率（全銘柄中 basis > 0 の割合）を計算する

**なぜ必要か**  
生basisのままでは「常に乖離が高い銘柄」が毎日ランク上位に固定される。  
Zスコア化することで「その銘柄にとって今日が特に割高/割安か」という相対的な比較ができる。

In [ ]:
# ============================================================
# Zスコア計算（銘柄ごとにbasisを標準化）
# ============================================================
def calc_zscore(group, window):
    """
    銘柄ごとに basis を 14日ローリングでZスコア化
    zscore = (basis - μ_14d) / σ_14d
    """
    b = group["basis"]
    mu = b.rolling(window, min_periods=window).mean()
    sd = b.rolling(window, min_periods=window).std()
    return (b - mu) / sd.replace(0, np.nan)


df["zscore"] = df.groupby("symbol", group_keys=False).apply(lambda g: calc_zscore(g, Z_WINDOW))
df_z = df.dropna(subset=["zscore", "next_ret"]).copy()

# BTC価格・方向性シグナル
btc = df[df["symbol"] == "BTC"][["date", "sp_close"]].set_index("date").sort_index()
btc_ret30 = btc["sp_close"].pct_change(BTC_WIN)

# コンタンゴ比率（全銘柄中 basis > 0 の割合）
contango_ratio = df_z.groupby("date")["basis"].apply(lambda x: (x > 0).mean())

print("Zスコア計算完了")
print(f"有効データ: {len(df_z):,}行")

# 戦略の仮説

1. **Zスコアが高い銘柄** = 先物が割高 = 収束する方向にショート
2. **Zスコアが低い銘柄** = 先物が割安 = 収束する方向にロング
3. **BTC相場環境で方向を切り替え** = 強気相場ではロング機会、弱気ではショート機会が増える
4. **コンタンゴ比率で市場全体の強弱を判定** = 弱気相場ではロングを止める

---
<a id="section1"></a>

### 📊 Step1: 生basis版バックテスト

**【問い】** Zスコア化する前の生basisで戦略を動かすと、どんな成績になるか？

**【仮説】** 生basisのままでは銘柄固有の「常に乖離が高い/低い」という特性が混ざり込み、正しいクロスセクション比較ができていないはず。まずベースラインとして成績を把握する。

**【検証方法】**
- 毎日、全銘柄をbasis降順でソート
- 上位10銘柄 → ショート（先物が現物より割高な銘柄を売る）
- 下位10銘柄 → ロング（先物が現物より割安な銘柄を買う）
- 翌日決済・手数料0.04%/片道

In [ ]:
# <a id="section1"></a>
# ============================================================
# Step1: 生basis版（ベースライン）
# ============================================================
rows = []
df_raw = df.dropna(subset=["basis", "next_ret"]).copy()

for date, group in df_raw.groupby("date"):
    valid = group.dropna(subset=["next_ret"])
    if len(valid) < TOP_N * 2:
        continue
    sg = valid.sort_values("basis", ascending=False)
    short_ret = (-sg.head(TOP_N)["next_ret"].mean()) - FEE
    long_ret = (sg.tail(TOP_N)["next_ret"].mean()) - FEE
    rows.append(
        {
            "date": date,
            "short_ret": short_ret,
            "long_ret": long_ret,
            "combined_ret": (short_ret + long_ret) / 2,
        }
    )

bt_raw = pd.DataFrame(rows)

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Step1: 生basis版バックテスト（2021〜2026）", fontsize=13, fontweight="bold")

ax1 = axes[0]
(bt_raw.set_index("date")["short_ret"].cumsum() * 100).plot(
    ax=ax1, label=f'ショート (Sharpe={sharpe(bt_raw["short_ret"]):+.3f})', color="#E74C3C", lw=1.8
)
(bt_raw.set_index("date")["long_ret"].cumsum() * 100).plot(
    ax=ax1, label=f'ロング  (Sharpe={sharpe(bt_raw["long_ret"]):+.3f})', color="#3498DB", lw=1.8
)
(bt_raw.set_index("date")["combined_ret"].cumsum() * 100).plot(
    ax=ax1,
    label=f'両サイド (Sharpe={sharpe(bt_raw["combined_ret"]):+.3f})',
    color="#27AE60",
    lw=2.5,
    ls="--",
)
ax1.axhline(0, color="gray", lw=1)
ax1.set_ylabel("累積リターン (%)", fontsize=11)
ax1.set_title("累積リターン", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

ax2 = axes[1]
bt_raw["year"] = bt_raw["date"].dt.year
years = sorted(bt_raw["year"].unique())
x = np.arange(len(years))
w = 0.28
ax2.bar(
    x - w,
    [sharpe(bt_raw[bt_raw["year"] == y]["short_ret"]) for y in years],
    width=w,
    label="ショート",
    color="#E74C3C",
    alpha=0.85,
)
ax2.bar(
    x,
    [sharpe(bt_raw[bt_raw["year"] == y]["long_ret"]) for y in years],
    width=w,
    label="ロング",
    color="#3498DB",
    alpha=0.85,
)
ax2.bar(
    x + w,
    [sharpe(bt_raw[bt_raw["year"] == y]["combined_ret"]) for y in years],
    width=w,
    label="両サイド",
    color="#27AE60",
    alpha=0.85,
)
ax2.axhline(0, color="black", lw=1.2)
ax2.set_xticks(x)
ax2.set_xticklabels(years)
ax2.set_ylabel("Sharpe比（年率）", fontsize=11)
ax2.set_title("年別 Sharpe比", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(
    f"生basis版 両サイド: Sharpe={sharpe(bt_raw['combined_ret']):+.3f}, 累積={bt_raw['combined_ret'].sum()*100:+.1f}%"
)

### Step1 結果まとめ

| サイド | Sharpe | 傾向 |
|--------|--------|------|
| ショート | -0.28 | ❌ ブル相場でショートが機能しない |
| ロング | +0.32 | △ 上昇はBTCベータに乗っているだけ |
| 両サイド | +0.11 | △ ほぼ意味なし |

**問題の本質**: 生basisでは銘柄間の比較が不公平（常に乖離が高い銘柄が固定ランク入り）

→ **Zスコア化が必要**

---
<a id="section2"></a>

### 📊 Step2: Zスコア化効果検証

**【問い】** basisをZスコア化するだけで、成績はどれだけ改善するか？

**【観察】** Step1（生basis版）では Sharpe -0.275。「常に乖離が高い銘柄」が毎日ランク入りする問題があり、公平な比較ができていなかった。

**【仮説】** 銘柄ごとに直近14日の平均・標準偏差で標準化することで、「その銘柄にとって今日が特に割高/割安か」という相対比較ができるようになり、信号の質が上がるはず。

**【検証方法】**
- 生basisの代わりにZスコアでランキングする（それ以外の条件は同じ）
- 上位10銘柄（Zスコア高 = 先物が相対的に割高）→ ショート
- 下位10銘柄（Zスコア低 = 先物が相対的に割安）→ ロング

In [ ]:
# <a id="section2"></a>
# ============================================================
# Step2: Zスコア版（標準化後のクロスセクション）
# ============================================================
rows = []
for date, group in df_z.groupby("date"):
    valid = group.dropna(subset=["next_ret"])
    if len(valid) < TOP_N * 2:
        continue
    sg = valid.sort_values("zscore", ascending=False)
    short_ret = (-sg.head(TOP_N)["next_ret"].mean()) - FEE
    long_ret = (sg.tail(TOP_N)["next_ret"].mean()) - FEE
    rows.append(
        {
            "date": date,
            "short_ret": short_ret,
            "long_ret": long_ret,
            "combined_ret": (short_ret + long_ret) / 2,
        }
    )

bt_z = pd.DataFrame(rows)
bt_z["year"] = bt_z["date"].dt.year

# 生basis vs Zスコア 比較
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Step2: 生basis版 vs Zスコア版 比較", fontsize=13, fontweight="bold")

ax1 = axes[0]
(bt_raw.set_index("date")["combined_ret"].cumsum() * 100).plot(
    ax=ax1,
    label=f'生basis (Sharpe={sharpe(bt_raw["combined_ret"]):+.3f})',
    color="#95A5A6",
    lw=2,
    ls="--",
    zorder=3,
)
(bt_z.set_index("date")["combined_ret"].cumsum() * 100).plot(
    ax=ax1,
    label=f'Zスコア (Sharpe={sharpe(bt_z["combined_ret"]):+.3f})',
    color="#27AE60",
    lw=2.5,
    zorder=4,
)

# BTC背景（右軸, 薄色）
ax1_btc = ax1.twinx()
btc_period = btc[(btc.index >= bt_raw["date"].min()) & (btc.index <= bt_raw["date"].max())][
    "sp_close"
].dropna()
if len(btc_period) > 1:
    btc_bg = (btc_period / btc_period.iloc[0] - 1.0) * 100
    ax1_btc.plot(
        btc_bg.index,
        btc_bg.values,
        color="#BDC3C7",
        lw=2.0,
        alpha=0.35,
        zorder=1,
        label="BTC spot（背景）",
    )
    ax1_btc.fill_between(btc_bg.index, btc_bg.values, 0, color="#BDC3C7", alpha=0.08, zorder=1)
ax1_btc.set_yticks([])
ax1_btc.set_ylabel("")
ax1_btc.grid(False)

ax1.axhline(0, color="gray", lw=1)
ax1.set_ylabel("累積リターン (%)", fontsize=11)
ax1.set_title("累積リターン比較（両サイド合成）", fontsize=11)

# 凡例を統合
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax1_btc.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, fontsize=10)
ax1.grid(alpha=0.3)

ax2 = axes[1]
x = np.arange(len(years))
w = 0.35
ax2.bar(
    x - w / 2,
    [sharpe(bt_raw[bt_raw["year"] == y]["combined_ret"]) for y in years],
    width=w,
    label="生basis",
    color="#95A5A6",
    alpha=0.85,
)
ax2.bar(
    x + w / 2,
    [sharpe(bt_z[bt_z["year"] == y]["combined_ret"]) for y in years],
    width=w,
    label="Zスコア",
    color="#27AE60",
    alpha=0.85,
)
ax2.axhline(0, color="black", lw=1.2)
ax2.set_xticks(x)
ax2.set_xticklabels(years)
ax2.set_ylabel("Sharpe比（年率）", fontsize=11)
ax2.set_title("年別 Sharpe比 比較", fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()
print(
    f"Zスコア版: Sharpe={sharpe(bt_z['combined_ret']):+.3f}（生basis比 {sharpe(bt_z['combined_ret'])-sharpe(bt_raw['combined_ret']):+.3f}改善）"
)

### 結果
Zスコア化により Sharpe **+0.38改善**（-0.275 → +0.108）<br>
ただし全期間通じて Sharpe +0.108 はまだ低い。<br>
ロング（Sharpe +0.32）はBTC上昇相場では機能するが、下落相場（2022・2025）では機能しない。<br>

→ **相場環境に応じて戦略を切り替える必要がある**

---
<a id="section3"></a>

### 📊 Step3: BTCスイッチング + IS/OOS検証

**【問い】** 相場の方向に合わせてロング/ショートを切り替えると、成績はどう変わるか？

**【観察】** Step2（Zスコア版）の年別成績を見ると、強気相場ではロングが機能しショートが損失を出す。弱気相場では逆転する。両方を毎日実行することで、相場環境と逆のポジションが足を引っ張っている。

**【仮説】** BTC の30日リターンを「相場センサー」として使い、強気相場ではロングのみ・弱気相場ではショートのみ実行することで、パフォーマンスが改善するはず。

**【検証方法】**
- BTC 30日リターン > +x% → 強気相場 → ロングのみ実行
- BTC 30日リターン < -x% → 弱気相場 → ショートのみ実行
- ±7%以内 → 方向感なし → スキップ
- バッファ閾値（±x%）はIS期間（2021-2022）のグリッドサーチで決定。OOSは「答え合わせ」として後で見る。

| 相場環境 | ロング | ショート | 原因 |
|---------|--------|---------|------|
| 強気相場（BTC上昇） | ✅ 機能する | ❌ 機能しない | アルトも全体的に上がるのでショートが損 |
| 弱気相場（BTC下落） | ❌ 機能しない | ✅ 機能する | アルトも全体的に下がるのでロングが損 |

---

### バッファ閾値をどう決めるか？ — IS期間のみでグリッドサーチ

「どこからプラス/マイナスと判断するか」のしきい値（バッファ）を決める。

**方針**: IS期間（2021-2022）のSharpeだけを見て決定する。  
候補として `±3% / ±5% / ±7% / ±10% / ±15%` を試す。

> ⚠️ **OOSの結果を見てから選ぶのはNG（過学習）**  
> 先にOOSを見て「成績が良くなる値」を選ぶと、未来データに合わせて調整したのと同じになる。  
> IS で決定し、OOS は「答え合わせ」として後で見る。

> 💡 **なぜ本体の実行より先にパラメータを決めるのか**  
> パラメータを決めてから戦略を動かすのが正しい順序。  
> 先に動かして後から「成績が良くなる値」を選ぶのは後付け最適化と同じ。

In [ ]:
# ============================================================
# バッファ閾値のグリッドサーチ
# ============================================================
buffer_list = [0.03, 0.05, 0.07, 0.10, 0.15]
grid_results = []

for buf in buffer_list:
    rows_g = []
    for date, group in df_z.groupby("date"):
        valid = group.dropna(subset=["next_ret"])
        if len(valid) < TOP_N * 2:
            continue
        if date not in btc_ret30.index:
            continue
        btc_r = btc_ret30.loc[date]
        if pd.isna(btc_r):
            continue
        if btc_r > buf:
            mode = "long"
        elif btc_r < -buf:
            mode = "short"
        else:
            continue
        sg = valid.sort_values("zscore", ascending=False)
        ret = (
            (sg.tail(TOP_N)["next_ret"].mean() - FEE)
            if mode == "long"
            else (-sg.head(TOP_N)["next_ret"].mean() - FEE)
        )
        rows_g.append({"date": date, "ret": ret})
    bt_g = pd.DataFrame(rows_g)
    s_is = sharpe(bt_g[bt_g["date"] <= IS_END]["ret"])
    s_oos = sharpe(bt_g[bt_g["date"] >= OOS_START]["ret"])
    grid_results.append({"buffer": f"±{buf:.0%}", "IS": s_is, "OOS": s_oos, "bt": bt_g})
    print(f"buffer={buf:.0%}  IS={s_is:+.3f}  OOS={s_oos:+.3f}")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("バッファ閾値のグリッドサーチ", fontsize=13, fontweight="bold")

# 左: IS/OOS Sharpe比較
ax1 = axes[0]
labels = [r["buffer"] for r in grid_results]
is_vals = [r["IS"] for r in grid_results]
oos_vals = [r["OOS"] for r in grid_results]
x = range(len(labels))
ax1.plot(x, is_vals, "o-", color="#3498DB", lw=2, markersize=8, label="IS (2021-2022)")
ax1.plot(x, oos_vals, "s-", color="#E74C3C", lw=2, markersize=8, label="OOS (2023-2026)")
best_idx = is_vals.index(max(is_vals))  # IS期間のみで最良値を選択（OOSは見ない）
ax1.axvline(best_idx, color="#27AE60", lw=2, ls="--", label=f"採用値: {labels[best_idx]}（IS最大）")
for i, (iv, ov) in enumerate(zip(is_vals, oos_vals)):
    ax1.annotate(
        f"{iv:+.2f}",
        (i, iv),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=8.5,
        color="#3498DB",
    )
    ax1.annotate(
        f"{ov:+.2f}",
        (i, ov),
        textcoords="offset points",
        xytext=(0, -15),
        ha="center",
        fontsize=8.5,
        color="#E74C3C",
    )
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels, fontsize=11)
ax1.set_ylabel("Sharpe比（年率）", fontsize=11)
ax1.set_title("バッファ別 IS/OOS Sharpe\n（採用値はIS期間のみで決定）", fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右: 採用値（7%）での累積リターン
ax2 = axes[1]
bt_best = grid_results[best_idx]["bt"]
s_is_best = bt_best[bt_best["date"] <= IS_END]
s_oos_best = bt_best[bt_best["date"] >= OOS_START]
ax2.plot(
    s_is_best.set_index("date")["ret"].cumsum() * 100,
    color="#3498DB",
    lw=2,
    label=f'IS  Sharpe={sharpe(s_is_best["ret"]):+.2f}',
)
ax2.plot(
    s_oos_best.set_index("date")["ret"].cumsum() * 100,
    color="#E74C3C",
    lw=2,
    label=f'OOS Sharpe={sharpe(s_oos_best["ret"]):+.2f}',
)
ax2.axhline(0, color="gray", lw=1)
ax2.set_ylabel("累積リターン (%)", fontsize=11)
ax2.set_title(f"採用値 {labels[best_idx]} での累積リターン\n（ISで選択 → OOSで検証）", fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

### ✅ パラメータ決定 → BTCスイッチング本検証へ

グリッドサーチの結果、**IS期間のSharpeが最も高かったバッファ ±7%** を採用した。

```
BTC 30日リターン > +7%  → 強気相場 → ロングのみ実行
BTC 30日リターン < -7%  → 弱気相場 → ショートのみ実行
±7%以内              → 方向感なし → スキップ
```

このパラメータを使って、IS（2021-2022）と OOS（2023-2026）に分けて本検証を行う。  
OOS は「答え合わせ」— ここで初めて見る。

In [ ]:
# <a id="section3"></a>
# ============================================================
# Step3: BTCスイッチング + IS/OOS検証
# ============================================================
# BTC 30日リターンで方向を判定
# > +7%  → ロングモード（アルトの割安銘柄を買う）
# < -7%  → ショートモード（アルトの割高銘柄を売る）
# ±7%以内 → スキップ（方向感なし）


def run_btc_switch(contango_thr=None):
    rows = []
    for date, group in df_z.groupby("date"):
        valid = group.dropna(subset=["next_ret"])
        if len(valid) < TOP_N * 2:
            continue
        if date not in btc_ret30.index:
            continue
        btc_r = btc_ret30.loc[date]
        if pd.isna(btc_r):
            continue

        if btc_r > BUFFER:
            mode = "long"
        elif btc_r < -BUFFER:
            mode = "short"
        else:
            continue

        sg = valid.sort_values("zscore", ascending=False)
        if mode == "short":
            ret = (-sg.head(TOP_N)["next_ret"].mean()) - FEE
        else:
            if contango_thr is not None:
                cr = contango_ratio.get(date, np.nan)
                if pd.isna(cr) or cr < contango_thr:
                    continue
            ret = (sg.tail(TOP_N)["next_ret"].mean()) - FEE
        rows.append({"date": date, "mode": mode, "ret": ret})

    bt = pd.DataFrame(rows)
    bt["year"] = bt["date"].dt.year
    return bt


bt_switch = run_btc_switch(contango_thr=None)

# IS/OOS分割で可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Step3: BTCスイッチング戦略（IS/OOS検証）", fontsize=13, fontweight="bold")

for ax, label, start, end, color in [
    (axes[0], "IS期間（2021-2022）", "2021-01-01", IS_END, "#3498DB"),
    (axes[1], "OOS期間（2023-2026）", OOS_START, "2026-12-31", "#E74C3C"),
]:
    sub = bt_switch[(bt_switch["date"] >= start) & (bt_switch["date"] <= end)]
    s = sub.set_index("date")["ret"]
    sp = sharpe(sub["ret"])
    dd = maxdd(sub["ret"])

    # BTC相場を背景に薄く表示（右軸、%騰落率に正規化）
    ax_btc = ax.twinx()
    btc_period = btc[(btc.index >= pd.Timestamp(start)) & (btc.index <= pd.Timestamp(end))][
        "sp_close"
    ].dropna()
    if len(btc_period) > 1:
        btc_bg = (btc_period / btc_period.iloc[0] - 1.0) * 100
        ax_btc.plot(btc_bg.index, btc_bg.values, color="#BDC3C7", lw=2.0, alpha=0.35, zorder=1)
        ax_btc.fill_between(btc_bg.index, btc_bg.values, 0, color="#BDC3C7", alpha=0.08, zorder=1)
    ax_btc.set_yticks([])
    ax_btc.set_ylabel("")
    ax_btc.grid(False)

    # 戦略累積リターン（前面）
    cum = s.cumsum() * 100
    ax.plot(cum.index, cum.values, color=color, lw=2, zorder=3)
    ax.fill_between(cum.index, cum.values, 0, alpha=0.15, color=color, zorder=2)
    ax.axhline(0, color="gray", lw=1, zorder=2)
    ax.set_title(f"{label}\nSharpe={sp:+.3f}  MaxDD={dd:.1f}%  取引日={len(sub)}日", fontsize=10)
    ax.set_ylabel("累積リターン (%)", fontsize=10)
    ax.text(
        0.02,
        0.95,
        "薄灰: BTC騰落率(背景)",
        transform=ax.transAxes,
        va="top",
        fontsize=8,
        color="#7f8c8d",
    )
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

is_sub = bt_switch[bt_switch["date"] <= IS_END]
oos_sub = bt_switch[bt_switch["date"] >= OOS_START]
print(f"IS  Sharpe: {sharpe(is_sub['ret']):+.3f}")
print(f"OOS Sharpe: {sharpe(oos_sub['ret']):+.3f}  ← OOSで大きく低下")

### Step3 結果

| 期間 | Sharpe | 取引日数 |
|------|--------|---------|
| IS（2021-2022） | +1.941 | 486日 |
| OOS（2023-2026） | +0.341 | 668日 |

**観察**: IS/OOSで大きな乖離。OOS期間（2025〜2026）は下げ相場でも苦戦。

**原因**: 2025〜2026は市場全体がバックワーデーション（先物割安）→ ロング戦略が機能しない  
→ **市場全体の強弱を判定するフィルターが必要**

---
<a id="section4"></a>

### 📊 Step4: コンタンゴフィルター追加

**【問い】** 市場全体が弱気の時はロングをスキップすると、OOS期間（2025〜2026年）の成績は改善するか？

**【仮説】** 全銘柄のコンタンゴ比率（basis > 0 の割合）を「市場の体温計」として使うことで、ロング環境か否かを判定できる。比率が低い時はロングをスキップすることで、弱気相場での損失を回避できるはず。

**【検証方法】**
- コンタンゴ比率 < 閾値 → ロングスキップ（市場全体が弱い）
- コンタンゴ比率 ≥ 閾値 → ロング実行
- 閾値はバッファと同様にIS期間のグリッドサーチで決定する

---

### コンタンゴ比率とは？

**定義**: 全371銘柄のうち、`basis > 0`（先物が現物より高い＝コンタンゴ）の銘柄が何割あるか。

```
コンタンゴ比率 = コンタンゴ銘柄数 / 全銘柄数
```

**イメージ**

| 市場の状態 | コンタンゴ比率 | 意味 |
|-----------|-------------|------|
| 強気相場（みんな買いたい） | 高い（60〜80%） | 多くの銘柄で先物プレミアムが乗っている |
| 弱気相場（みんな売りたい） | 低い（10〜20%） | ほとんどの銘柄で先物が現物を下回っている |

**実際のデータで見ると：**

| 年 | コンタンゴ比率（年平均） | 相場 |
|----|----------------------|------|
| 2021 | 63.6% | 強気 📈 |
| 2022 | 11.9% | 弱気 📉 |
| 2023 | 29.7% | 回復途上 |
| 2024 | 40.3% | 強気 📈 |
| 2025 | 13.2% | 弱気 📉 |
| 2026 | 12.7% | 弱気継続 📉 |

---

**なぜ比率が低い時にロングしてはいけないか？**

ロング戦略の仮説は「先物が割安 → いずれ収束して上がる」というもの。  
しかし市場全体がバックワーデーション（先物割安）の時は、  
「割安なのではなく、みんなが先物を売っている（下落への備え）」という状態。  
この時に「割安だからロング」しても、そのまま下がり続けることが多い。

---

**追加するフィルター**  
```
ロングモードの日に：
  コンタンゴ比率 < 閾値  → スキップ（市場全体が弱い、ロングしない）
  コンタンゴ比率 ≥ 閾値  → ロング実行
```

→ **閾値をいくつに設定するかは、バッファと同様にIS期間のグリッドサーチで決定する。**

---

### コンタンゴ閾値をどう決めるか？ — IS期間のみでグリッドサーチ

バッファ（±7%）と同じ考え方で、**IS期間（2021-2022）のSharpeだけを見て決定する。**  
候補として `15% / 20% / 25% / 30% / 35% / 40%` を試す。

> ⚠️ **OOSの結果を見てから選ぶのはNG（過学習）**  
> IS で決定し、OOS は「答え合わせ」として後で見る。

In [ ]:
# ============================================================
# コンタンゴ閾値のグリッドサーチ（IS期間のみで決定）
# ============================================================
contango_thr_list = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
contango_grid_results = []

for thr in contango_thr_list:
    bt_g = run_btc_switch(contango_thr=thr)
    s_is = sharpe(bt_g[bt_g["date"] <= IS_END]["ret"])
    s_oos = sharpe(bt_g[bt_g["date"] >= OOS_START]["ret"])
    contango_grid_results.append({"thr": f"{thr:.0%}", "IS": s_is, "OOS": s_oos, "bt": bt_g})
    print(f"thr={thr:.0%}  IS={s_is:+.3f}  OOS={s_oos:+.3f}")

# 可視化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("コンタンゴ閾値のグリッドサーチ（IS期間のみで決定）", fontsize=13, fontweight="bold")

labels_c = [r["thr"] for r in contango_grid_results]
is_vals_c = [r["IS"] for r in contango_grid_results]
oos_vals_c = [r["OOS"] for r in contango_grid_results]
best_idx_c = is_vals_c.index(max(is_vals_c))
x = range(len(labels_c))

ax1 = axes[0]
ax1.plot(x, is_vals_c, "o-", color="#3498DB", lw=2, markersize=8, label="IS (2021-2022)")
ax1.plot(x, oos_vals_c, "s-", color="#E74C3C", lw=2, markersize=8, label="OOS (2023-2026)")
ax1.axvline(
    best_idx_c, color="#27AE60", lw=2, ls="--", label=f"採用値: {labels_c[best_idx_c]}（IS最大）"
)
for i, (iv, ov) in enumerate(zip(is_vals_c, oos_vals_c)):
    ax1.annotate(
        f"{iv:+.2f}",
        (i, iv),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center",
        fontsize=9,
        color="#3498DB",
    )
    ax1.annotate(
        f"{ov:+.2f}",
        (i, ov),
        textcoords="offset points",
        xytext=(0, -15),
        ha="center",
        fontsize=9,
        color="#E74C3C",
    )
ax1.set_xticks(list(x))
ax1.set_xticklabels(labels_c, fontsize=11)
ax1.set_ylabel("Sharpe比（年率）", fontsize=11)
ax1.set_title("閾値別 IS/OOS Sharpe\n（採用値はIS期間のみで決定）", fontsize=11)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右: 採用値での累積リターン（IS/OOS）
ax2 = axes[1]
bt_best_c = contango_grid_results[best_idx_c]["bt"]
s_is_c = bt_best_c[bt_best_c["date"] <= IS_END]
s_oos_c = bt_best_c[bt_best_c["date"] >= OOS_START]
ax2.plot(
    s_is_c.set_index("date")["ret"].cumsum() * 100,
    color="#3498DB",
    lw=2,
    label=f'IS  Sharpe={sharpe(s_is_c["ret"]):+.2f}',
)
ax2.plot(
    s_oos_c.set_index("date")["ret"].cumsum() * 100,
    color="#E74C3C",
    lw=2,
    label=f'OOS Sharpe={sharpe(s_oos_c["ret"]):+.2f}',
)
ax2.axhline(0, color="gray", lw=1)
ax2.set_ylabel("累積リターン (%)", fontsize=11)
ax2.set_title(
    f"採用値 {labels_c[best_idx_c]} での累積リターン\n（ISで選択 → OOSで検証）", fontsize=11
)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

best_thr = contango_thr_list[best_idx_c]
print(f"\n採用値: コンタンゴ閾値 = {best_thr:.0%}（IS Sharpe最大: {is_vals_c[best_idx_c]:+.3f}）")

---

### ✅ コンタンゴ閾値決定 → 最終版検証へ

グリッドサーチの結果、**IS期間のSharpeが最も高かった閾値 20% を採用**した（IS Sharpe: +2.16）。

```
ロングモードの日に：
  コンタンゴ比率 < 20%  → スキップ（市場全体が弱い、ロングしない）
  コンタンゴ比率 ≥ 20%  → ロング実行
```

このパラメータで Step3（BTCスイッチングのみ）と比較し、コンタンゴフィルターの効果を確認する。

In [ ]:
# <a id="section4"></a>
# ============================================================
# Step4: コンタンゴフィルター追加
# ============================================================
# コンタンゴ比率 = 全銘柄中 basis > 0 の割合
# この比率が低い（< 20%）= 市場全体が弱気 → ロングをスキップ

bt_final = run_btc_switch(contango_thr=CONTANGO_THR)

# IS/OOS比較（スイッチのみ vs 最終版）
fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle("Step4: コンタンゴフィルター追加（最終版）", fontsize=13, fontweight="bold")

# 上段: IS/OOS累積リターン（背景BTCあり）
for ax, start, end, title in [
    (axes[0][0], "2021-01-01", IS_END, "IS期間（2021-2022）"),
    (axes[0][1], OOS_START, "2026-12-31", "OOS期間（2023-2026）"),
]:
    # 1) 戦略線
    strategy_series = {}
    for bt, label, color, ls in [
        (bt_switch, "BTCスイッチのみ", "#95A5A6", "--"),
        (bt_final, "+コンタンゴ20%", "#27AE60", "-"),
    ]:
        sub = bt[(bt["date"] >= start) & (bt["date"] <= end)].copy()
        s = sub.set_index("date")["ret"].sort_index()
        cum = s.cumsum() * 100
        strategy_series[label] = (s, cum)
        ax.plot(
            cum.index,
            cum.values,
            label=f'{label} (Sharpe={sharpe(sub["ret"]):+.2f})',
            color=color,
            lw=2,
            ls=ls,
            zorder=2,
        )

    # 2) 最終版（+コンタンゴ20%）のエントリー点描（entryのみ）
    if "+コンタンゴ20%" in strategy_series:
        _, cum_fin = strategy_series["+コンタンゴ20%"]
        if len(cum_fin) > 0:
            entry_y = cum_fin.shift(1).fillna(0)
            ax.scatter(
                cum_fin.index,
                entry_y.values,
                s=14,
                marker="o",
                facecolors="none",
                edgecolors="#1F77B4",
                alpha=0.55,
                linewidths=0.9,
                zorder=3,
                label="entry",
            )

    # 3) BTC背景（右軸, 薄色）
    ax_btc = ax.twinx()
    btc_period = btc[(btc.index >= pd.Timestamp(start)) & (btc.index <= pd.Timestamp(end))][
        "sp_close"
    ].dropna()
    if len(btc_period) > 1:
        btc_bg = (btc_period / btc_period.iloc[0] - 1.0) * 100
        ax_btc.plot(
            btc_bg.index,
            btc_bg.values,
            color="#BDC3C7",
            lw=2.0,
            alpha=0.35,
            zorder=1,
            label="BTC spot（背景）",
        )
        ax_btc.fill_between(btc_bg.index, btc_bg.values, 0, color="#BDC3C7", alpha=0.08, zorder=1)
    ax_btc.set_yticks([])
    ax_btc.set_ylabel("")
    ax_btc.grid(False)

    ax.axhline(0, color="gray", lw=1)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("累積リターン (%)", fontsize=10)
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax_btc.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=9)
    ax.grid(alpha=0.3)

# 下左: 年別Sharpe比較
ax3 = axes[1][0]
all_years = sorted(set(bt_switch["year"].tolist() + bt_final["year"].tolist()))
x = np.arange(len(all_years))
w = 0.35
ax3.bar(
    x - w / 2,
    [sharpe(bt_switch[bt_switch["year"] == y]["ret"]) for y in all_years],
    width=w,
    label="BTCスイッチのみ",
    color="#95A5A6",
    alpha=0.85,
)
ax3.bar(
    x + w / 2,
    [sharpe(bt_final[bt_final["year"] == y]["ret"]) for y in all_years],
    width=w,
    label="+コンタンゴ20%",
    color="#27AE60",
    alpha=0.85,
)
ax3.axhline(0, color="black", lw=1.2)
ax3.axvline(1.5, color="orange", lw=1.8, ls=":", label="IS/OOS境界")
ax3.set_xticks(x)
ax3.set_xticklabels([f"{y}\n(IS)" if y <= 2022 else f"{y}\n(OOS)" for y in all_years], fontsize=9)
ax3.set_ylabel("Sharpe比（年率）", fontsize=10)
ax3.set_title("年別 Sharpe比", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(axis="y", alpha=0.3)

# 下右: コンタンゴ比率推移
ax4 = axes[1][1]
cr_plot = contango_ratio.copy()
cr_plot.index = pd.to_datetime(cr_plot.index)
cr_roll = cr_plot.rolling(30).mean()
ax4.fill_between(cr_plot.index, cr_plot.values * 100, alpha=0.25, color="#3498DB", label="日次")
ax4.plot(cr_roll.index, cr_roll.values * 100, color="#2980B9", lw=2, label="30日MA")
ax4.axhline(20, color="#E74C3C", lw=2, ls="--", label="閾値 20%")
ax4.axvline(pd.Timestamp(OOS_START), color="orange", lw=2, ls=":", label="IS/OOS境界")
for yr, avg in [(2021, 63.6), (2022, 11.9), (2023, 29.7), (2024, 40.3), (2025, 13.2), (2026, 12.7)]:
    ax4.text(
        pd.Timestamp(f"{yr}-07-01"),
        88,
        f"{yr}\n{avg:.0f}%",
        ha="center",
        fontsize=8,
        fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"),
    )
ax4.set_ylabel("コンタンゴ銘柄の割合 (%)", fontsize=10)
ax4.set_title("コンタンゴ比率推移", fontsize=11)
ax4.legend(fontsize=9, loc="lower right")
ax4.grid(alpha=0.3)
ax4.set_ylim(0, 100)

plt.tight_layout()
plt.show()

# OOS期間のみ単体表示（entry点のみ）
fig_oos, ax_oos = plt.subplots(1, 1, figsize=(12, 5))
start, end = OOS_START, "2026-12-31"

# 左軸: 戦略累積リターン（+コンタンゴ20%のみ表示）
sub = bt_final[(bt_final["date"] >= start) & (bt_final["date"] <= end)].copy()
s = sub.set_index("date")["ret"].sort_index()
cum = s.cumsum() * 100
ax_oos.plot(
    cum.index,
    cum.values,
    label=f'+コンタンゴ20% (Sharpe={sharpe(sub["ret"]):+.2f})',
    color="#27AE60",
    lw=2,
    ls="-",
    zorder=3,
)

if len(cum) > 0:
    entry_y = cum.shift(1).fillna(0)
    ax_oos.scatter(
        cum.index,
        entry_y.values,
        s=16,
        marker="o",
        facecolors="none",
        edgecolors="#1F77B4",
        alpha=0.6,
        linewidths=1.0,
        zorder=4,
        label="entry",
    )

# 右軸: BTC価格（背景, 別スケール）
ax_btc = ax_oos.twinx()
btc_sub = btc[(btc.index >= pd.Timestamp(start)) & (btc.index <= pd.Timestamp(end))][
    "sp_close"
].dropna()
if len(btc_sub) > 1:
    ax_btc.plot(
        btc_sub.index,
        btc_sub.values,
        color="#BDC3C7",
        lw=2.0,
        alpha=0.35,
        zorder=1,
        label="BTC spot（右軸）",
    )

ax_oos.axhline(0, color="gray", lw=1)
ax_oos.set_title("Step4: OOS期間（2023-2026）累積リターン + entry点描", fontsize=12)
ax_oos.set_ylabel("累積リターン (%)", fontsize=10)
ax_oos.grid(alpha=0.3)

ax_btc.set_ylabel("BTC価格 (USDT)", fontsize=10, color="#7f8c8d")
ax_btc.tick_params(axis="y", colors="#7f8c8d")

# 凡例を統合
h1, l1 = ax_oos.get_legend_handles_labels()
h2, l2 = ax_btc.get_legend_handles_labels()
ax_oos.legend(h1 + h2, l1 + l2, fontsize=9, loc="upper left")

plt.tight_layout()
plt.show()

### Step4 結果まとめ

**コンタンゴ比率** = 全銘柄中 basis > 0 の割合。これが市場の「体温計」として機能した。

| 年 | コンタンゴ比率（年平均） | 市場環境 |
|----|----------------------|---------|
| 2021 | 63.6% | 強気相場 → ロング機能 |
| 2022 | 11.9% | 弱気相場 → ロングほぼ不可 |
| 2023 | 29.7% | 回復途上 → 一部ロング可 |
| 2024 | 40.3% | 強気相場 → ロング機能 |
| 2025 | 13.2% | 弱気相場 → ロングほぼ不可 |
| 2026 | 12.7% | 弱気継続 → ロング全スキップ |

フィルター適用後：OOS Sharpe **+0.341 → +0.90**（+0.56改善）

---
### OOS: ロング抑制期間の可視化（コンタンゴ<20%）

- **赤背景**: BTCはロングモード（`BTC30日 > +7%`）だが、コンタンゴ比率が20%未満のためロングを抑制した期間
- OOS（2023-2026）のみ表示


In [ ]:
# ============================================================
# OOSのみ: コンタンゴ比率フィルターでロング抑制された期間を可視化
# ============================================================
start, end = pd.Timestamp(OOS_START), pd.Timestamp("2026-12-31")

# 最終版の累積（OOS）
sub_oos = bt_final[(bt_final["date"] >= start) & (bt_final["date"] <= end)].copy()
s_oos = sub_oos.set_index("date")["ret"].sort_index()
cum_oos = s_oos.cumsum() * 100

# 抑制判定: BTCロングモードなのに、コンタンゴ比率不足でロングしない日
# long mode: BTC30日リターン > +7%
btc_long_mode = btc_ret30 > BUFFER
# contango ratio < 20%
contango_weak = contango_ratio < CONTANGO_THR

flag = (btc_long_mode & contango_weak).copy()
flag = flag[(flag.index >= start) & (flag.index <= end)]
flag = flag.sort_index()

# 連続True区間を [start, end] のspanへ変換
spans = []
in_span = False
span_start = None
prev_t = None
for t, v in flag.items():
    if bool(v) and not in_span:
        in_span = True
        span_start = t
    if in_span and (not bool(v)):
        spans.append((span_start, prev_t))
        in_span = False
        span_start = None
    prev_t = t
if in_span and span_start is not None:
    spans.append((span_start, prev_t))

fig, ax = plt.subplots(1, 1, figsize=(12, 5))

# 抑制期間を背景色で表示
for a, b in spans:
    ax.axvspan(a, b, color="#E74C3C", alpha=0.12)

# 最終版累積
ax.plot(
    cum_oos.index,
    cum_oos.values,
    color="#27AE60",
    lw=2,
    label=f'+コンタンゴ20% (Sharpe={sharpe(sub_oos["ret"]):+.2f})',
    zorder=3,
)

# entry点
if len(cum_oos) > 0:
    entry_y = cum_oos.shift(1).fillna(0)
    ax.scatter(
        cum_oos.index,
        entry_y.values,
        s=14,
        marker="o",
        facecolors="none",
        edgecolors="#1F77B4",
        alpha=0.55,
        linewidths=0.9,
        zorder=4,
        label="entry",
    )

# BTC背景（右軸）
ax_btc = ax.twinx()
btc_sub = btc[(btc.index >= start) & (btc.index <= end)]["sp_close"].dropna()
if len(btc_sub) > 1:
    ax_btc.plot(
        btc_sub.index,
        btc_sub.values,
        color="#BDC3C7",
        lw=2.0,
        alpha=0.35,
        label="BTC spot（右軸）",
        zorder=1,
    )

ax.axhline(0, color="gray", lw=1)
ax.set_title("Step4: OOS期間 ロング抑制期間ハイライト（赤背景）", fontsize=12)
ax.set_ylabel("累積リターン (%)", fontsize=10)
ax.grid(alpha=0.3)

ax_btc.set_ylabel("BTC価格 (USDT)", fontsize=10, color="#7f8c8d")
ax_btc.tick_params(axis="y", colors="#7f8c8d")

# 凡例統合 + 赤背景説明
import matplotlib.patches as mpatches

patch = mpatches.Patch(
    color="#E74C3C", alpha=0.12, label="ロング抑制期間（BTC>+7% かつ コンタンゴ<20%）"
)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax_btc.get_legend_handles_labels()
ax.legend([patch] + h1 + h2, [patch.get_label()] + l1 + l2, fontsize=9, loc="upper left")

plt.tight_layout()
plt.show()

print(f"抑制区間数: {len(spans)}")
if len(spans):
    total_days = int(sum((b - a).days + 1 for a, b in spans))
    print(f"抑制日数（概算）: {total_days}日")

<a id="summary"></a>

---
# まとめ

### 戦略の進化まとめ

| Step | 改善内容 | OOS Sharpe | 変化 |
|------|---------|-----------|------|
| 1 | 生basis版（ベースライン） | -0.275 | — |
| 2 | **Zスコア化**（銘柄間の公平な比較） | +0.108 | +0.38 ↑ |
| 3 | **BTCスイッチング**（方向性に合わせてロング/ショートを選択） | +0.341 | +0.23 ↑ |
| 4 | **コンタンゴフィルター**（市場全体が弱気の時はロングしない） | **+0.90** | **+0.56 ↑** |

### 学んだこと？

**① 標準化しないと銘柄間の比較ができない**  
生basisでは銘柄固有の特性が混在する。Zスコア化により「今日その銘柄が特に割高/割安か」を正しく比較できる。

**② IS/OOSで分けないと過学習に気づけない**  
過去データで最適化しても未来に通用しない可能性がある。学習期間（IS）と検証期間（OOS）を分けることで「本当に機能するか」を確認できる。

**③ 市場環境が変わったら戦略も変える**  
どんな戦略も相場環境が変わると機能しなくなる。コンタンゴ比率という「市場の体温計」を使うことで、ロングが機能しない弱気相場を事前に検知できた。

### 次の案

| 課題 | 内容 |
|------|------|
| 動的しきい値 | BTC判定の±7%バッファを固定値でなくボラティリティで動的に変える |

## 免責事項
* 本資料の実行・利用により生成または保存されるデータの管理は利用者の責任で行ってください。
* お客様によるコンテンツの利用等に関して生じうるいかなる損害について責任を負いません。
* 執筆者によって提供されたいかなる見解または意見は当該執筆者自身のその時点における見解や分析であって、当社の見解、分析ではありません。
* 暗号資産（仮想通貨）は法定通貨ではありません。
* また、法定通貨とは異なり、日本円やドルなどのように国又は特定の者によりその価値を保証されているものではありません。
* 暗号資産の価格の変動等により損失が発生する可能性があります。
* 暗号資産は代価の弁済を受ける者の同意がある場合に限り、代価の弁済のために使用することができます。
* 暗号資産信用取引は、価格の変動等により当初差入れた保証金を上回る損失が発生する可能性があります。十分なご理解の上で、自己責任にてお取引ください。
* お取引を行う際には、弊社のWebサイトに記載の「契約締結前交付書面兼説明書」「各種規約」「取引ルール」をご確認のうえ、取引内容を十分に理解し、お客様ご自身の責任と判断を持って行ってください。